# Notebook 17 detached workflow definitions

Purpose: launch a bounded training-only operator and complete JEPA engineering gate, then a four-pair learning screen only after efficiency passes, and fresh-seed statistical confirmation only after screening passes. This is executable project orchestration in tagged notebook cells. Run All here defines APIs without launching work; the frontdoor notebook starts actual detached tmux work.

Inputs: completed frozen Notebook15 ViT checkpoints, unchanged project exports, Food101 train/development membership and a single run ID. Outputs: source-bound phase clocks, latest-two/best full checkpoints, timing journals, receipts and visual report under `artifacts/notebook_17/<run_id>/`; checkpoints under `checkpoints/<run_id>/`.

Hard phase caps: feasibility2h, conditional screen3h at256px or4h at512px, conditional confirmation12h. A phase activates once and its wall clock cannot reset across resume. Exact resume: fresh `ahr-go` kernel, unchanged source exports, same run ID and frontdoor Run All. Invalid source/clock/checkpoint or interrupted active-wall journal yields UNKNOWN; never silently restart. Both long model workers run in detached tmux. Guard heartbeat10s, checkpoint/source review hourly, descendant cleanup on missing coordinator, cutoff or corruption. Final official pixels remain closed until all registered weights and gate receipts seal.

Resume cell: `launch-fixed-kernel-study` in `notebooks/17_Fixed_Kernel_Feasibility_and_Learning.ipynb`. Frontdoor consumer code is bound independently of tagged exports.

In [ ]:
"""Notebook17 detached, phase-bounded feasibility, learning screen and confirmation."""
from pathlib import Path
import copy,json,math,os,shlex,signal,subprocess,time,traceback
import nbformat
import numpy as np
from scipy import stats
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/17_fixed_kernel/workflow.ipynb'))
_namespace={'__file__':str(_SELF.with_name('common.ipynb'))}
for _cell in nbformat.read(_SELF.with_name('common.ipynb'),4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_SELF.with_name('common.ipynb')),'exec'),_namespace)
C=type('Common',(),_namespace)
MAIN='notebooks/17_Fixed_Kernel_Feasibility_and_Learning.ipynb'
TERMINAL={'COMPLETE','NO_GO','UNKNOWN','FAILED','VALIDATION_COMPLETE'}
PHASE_CAPS={'feasibility':7200.,'screen256':10800.,'screen512':14400.,'confirmation':43200.}

def artifact_path(root,run_id):
    if not isinstance(run_id,str)or not run_id or any(not(c.isalnum()or c in'-_')for c in run_id):raise ValueError('Safe stable RUN_ID required')
    return Path(root).resolve()/'artifacts/notebook_17'/run_id

def session_names(run_id):
    prefix='fmm17-'+run_id;return {'coordinator':prefix,'guard':prefix+'-guard','worker0':prefix+'-worker0','worker1':prefix+'-worker1'}

def session_exists(name):return subprocess.run(['tmux','has-session','-t','='+name],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0

def terminate_session_tree(session):
    """Stop only processes descending from this exact project-owned tmux session."""
    if not session_exists(session):return []
    roots={int(x)for x in subprocess.check_output(['tmux','list-panes','-t','='+session,'-F','#{pane_pid}'],text=True).split()}
    edges=[tuple(map(int,line.split()))for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()];owned=set(roots)
    while True:
        expanded=owned|{pid for pid,parent in edges if parent in owned}
        if expanded==owned:break
        owned=expanded
    for pid in sorted(owned-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t','='+session],capture_output=True);return sorted(owned)

def validate_manifest_rotation(runner,directory):
    for attempt in range(4):
        try:return runner.validate_training_manifest(directory)
        except (FileNotFoundError,ValueError):
            if attempt==3:raise
            time.sleep(.25)

def consumer_hash(root):return C.digest([''.join(c['source'])for c in json.loads((Path(root)/MAIN).read_text())['cells']if c['cell_type']=='code'])

def status(artifact,state,phase,**details):
    value={'status':state,'phase':phase,'updated_unix':time.time(),**details};C.atomic_json(Path(artifact)/'study_status.json',value);print(state,phase,flush=True);return value

def write_operator_notebook(root,path,function,arguments):
    code="from pathlib import Path\nimport json,nbformat,types,os\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nROOT=Path("+repr(str(Path(root).resolve()))+")\nP=ROOT/'experiments/17_fixed_kernel/workflow.ipynb'\nW=types.ModuleType('fmm17_detached_workflow');W.__file__=str(P)\nfor cell in nbformat.read(P,4).cells:\n    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)\nRESULT=W."+function+"(**json.loads("+repr(json.dumps(arguments))+"))\nprint(json.dumps(RESULT,indent=2,default=str))\n"
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Detached17 '+function+'\nGenerated operator entrypoint. Frozen parent source, original deadline and single-writer checkpoints remain authoritative. Resume through Notebook17 with the same RUN_ID.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR FMM-JEPA','language':'python'}})
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        old=nbformat.read(path,4)
        if [c.source for c in old.cells if c.cell_type=='code']!=[c.source for c in nb.cells if c.cell_type=='code']:raise ValueError('Detached entrypoint code changed')
    else:nbformat.write(nb,path)
    return path

def spawn(root,path,session,log):
    if session_exists(session):return {'status':'EXISTING_SESSION','session':session}
    jupyter=Path(root).parent/'.venv/bin/jupyter';command=' '.join(shlex.quote(str(x))for x in[jupyter,'nbconvert','--to','notebook','--execute','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',path])+' >> '+shlex.quote(str(log))+' 2>&1'
    env='CUBLAS_WORKSPACE_CONFIG=:4096:8 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 '
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),env+command],check=True)
    return {'status':'STARTED','session':session,'operator_notebook':str(path),'log':str(log)}

def _modules(root):
    return tuple(C.load(root,'experiments/17_fixed_kernel/'+name+'.ipynb','fmm17_workflow_'+name)for name in('benchmark','pipeline','runner','confirmation'))


def check_registration(root,artifact):
    req=C.read(Path(artifact)/'launch_registration.json')
    if req['sources']!=C.source_hashes(root)or req['consumer_hash']!=consumer_hash(root):raise ValueError('Frozen17 source/frontdoor changed; exact resume refused')
    # Phase activation publishes two small atomic records. Allow only a short
    # read retry for their publication boundary; clocks themselves never change.
    for attempt in range(6):
        journal=C.read(Path(artifact)/'phase_activations.json');active=C.read(Path(artifact)/'active_phase.json')
        if journal['activations']and active==journal['activations'][-1]:break
        if attempt==5:raise ValueError('Active phase differs from original activation journal')
        time.sleep(.05)
    if journal['binding']!=C.digest(req):raise ValueError('Phase clock registration binding differs')
    prior=[]
    for entry in journal['activations']:
        clock=C.read(entry['clock_record']['path'])
        if C.file_record(entry['clock_record']['path'])!=entry['clock_record']or clock!=entry['clock']:raise ValueError('Original phase clock changed; never reset or extend')
        phase=clock['phase'];cap=float(clock['maximum_seconds'])
        if phase not in PHASE_CAPS or cap!=PHASE_CAPS[phase]or abs(clock['deadline_unix']-clock['started_unix']-cap)>1e-5:raise ValueError('Registered bounded phase duration differs')
        if clock['binding']!=C.digest(req):raise ValueError('Phase clock source/consumer binding differs')
        if phase in prior:raise ValueError('Phase activated twice; no clock resets')
        prior.append(phase)
    if prior[0]!='feasibility'or len(prior)>3 or any(p not in('screen256','screen512')for p in prior[1:2])or(len(prior)==3 and prior[-1]!='confirmation'):raise ValueError('Invalid phase activation order')
    return req,active['clock']


def activate_phase(root,artifact,phase):
    artifact=Path(artifact);req=C.read(artifact/'launch_registration.json');path=artifact/'phases'/phase/'clock.json'
    journal=C.read(artifact/'phase_activations.json')if(artifact/'phase_activations.json').exists()else{'schema':'notebook17-phase-clock-journal/v1','binding':C.digest(req),'activations':[]}
    existing=[e for e in journal['activations']if e['clock']['phase']==phase]
    if existing:
        if journal['activations'][-1]!=existing[0]:raise ValueError('Cannot reactivate earlier phase')
        return check_registration(root,artifact)[1]
    if path.exists():raise ValueError('Orphan original phase clock requires inspection; never silently restart')
    if phase=='feasibility'and journal['activations']:raise ValueError('Feasibility clock already activated')
    if phase.startswith('screen')and [e['clock']['phase']for e in journal['activations']]!=['feasibility']:raise ValueError('Screen must follow feasibility once')
    if phase=='confirmation'and(len(journal['activations'])!=2 or not journal['activations'][-1]['clock']['phase'].startswith('screen')):raise ValueError('Confirmation must follow screened candidate')
    started=time.time();clock={'schema':'notebook17-immutable-phase-clock/v1','phase':phase,'started_unix':started,'deadline_unix':started+PHASE_CAPS[phase],
        'maximum_seconds':PHASE_CAPS[phase],'binding':C.digest(req),'restart_policy':'Original elapsed wall clock persists across interruption; no extension',
        'scope':'Whole phase including source verification, data, exact-resume proofs, model work, checkpoint IO and all phase readouts'}
    C.immutable(path,clock);entry={'clock':clock,'clock_record':C.file_record(path)};journal['activations'].append(entry)
    C.atomic_json(artifact/'phase_activations.json',journal);C.atomic_json(artifact/'active_phase.json',entry);return clock


def report(root,artifact,result,bench=None,cost=None,pipeline=None,screen=None):
    module=C.load(root,'experiments/17_fixed_kernel/report.ipynb','fmm17_report')
    return module.render(root,artifact,result,bench=bench,cost=cost,pipeline=pipeline,screen=screen)


def run_validation(root,run_id='fixed-kernel-validation'):
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    result={'status':'VALIDATION_COMPLETE','fixture_source':'SOFTWARE_PRESENTATION_FIXTURE_ONLY; NOT SCIENTIFIC RESULTS','run_id':run_id,
        'official_test':'CLOSED','long_runs_launched':False,'source_hashes':C.source_hashes(root),
        'reason':'Fresh Run All wiring and visual report only; independent real operator/checkpoint tests have separate receipts.'}
    report(root,artifact,result);status(artifact,'VALIDATION_COMPLETE','software_presentation_only');return result


def launch_study(root,run_id='fixed-kernel-001',profile=None,source_run='vit-geometry-001'):
    root=Path(root).resolve();C.bootstrap(root);profile=profile or os.environ.get('FMM17_PROFILE','full')
    if profile=='validation':return run_validation(root,run_id)
    if profile!='full':raise ValueError('Registered full or validation profile required')
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    if source_run!='vit-geometry-001':raise ValueError('This registered study freezes source_run=vit-geometry-001; a changed source needs a separately bound design')
    from fmm_jepa.food101_training import run_writer
    with run_writer(artifact/'launch_writer'):
        regpath=artifact/'launch_registration.json'
        if regpath.exists():
            req,clock=check_registration(root,artifact)
            if req['source_run']!=source_run:raise ValueError('Existing run source request differs')
        else:
            if (artifact/'phase_activations.json').exists()or(artifact/'active_phase.json').exists():raise ValueError('Orphan phase clock; never restart')
            req={'schema':'notebook17-launch/v1','run_id':run_id,'profile':'full','source_run':source_run,'sources':C.source_hashes(root),'consumer_hash':consumer_hash(root),
                'reference':C.handoff(root,source_run),'phase_caps':PHASE_CAPS,'official_test':'CLOSED','selection':'Training-only rank/pipeline/cost gates, then one four-pair learning screen; fresh power-derived final confirmation only after all gates pass'}
            C.immutable(regpath,req);clock=activate_phase(root,artifact,'feasibility')
        existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
        if existing.get('status')in TERMINAL:return {**existing,'report':str(artifact/'report.html'),'resume':'Terminal receipt retained; changed research design requires a new run ID'}
        C.check_deadline(clock['deadline_unix']);sessions=session_names(run_id)
        if not(artifact/'report.html').exists():
            report(root,artifact,{'status':'RUNNING','run_id':run_id,'reason':'Detached launch is being prepared; no scientific measurements completed yet',
                'active_phase_clock':clock,'phase_caps':PHASE_CAPS,'official_test':'CLOSED','source_hashes':req['sources']})
        guard=write_operator_notebook(root,artifact/'operators/guard.ipynb','run_guard',{'root':str(root),'run_id':run_id})
        coordinator=write_operator_notebook(root,artifact/'operators/coordinator.ipynb','run_coordinator',{'root':str(root),'run_id':run_id})
        guards=spawn(root,guard,sessions['guard'],artifact/'guard.log');job=spawn(root,coordinator,sessions['coordinator'],artifact/'coordinator.log')
        value={'status':'DETACHED_PIPELINE_STARTED','run_id':run_id,'sessions':sessions,'active_phase_clock':clock,'phase_caps':PHASE_CAPS,
            'conditional_maximum_seconds':7200+14400+43200,'artifact':str(artifact),'attach':'tmux attach -t '+sessions['coordinator'],'detach':'Ctrl-b, then d',
            'resume':'Fresh ahr-go kernel, same RUN_ID, Notebook17 Run All. Original phase clocks, full-state checkpoints and timer journals must remain valid; IN_FLIGHT timing interruption stays UNKNOWN.',
            'report':str(artifact/'report.html'),'guard':guards,'coordinator':job}
        C.atomic_json(artifact/'launch_receipt.json',value);return value


def _receipt(role,path,passed,scope):
    return {'role':role,'status':'PASS'if passed else'FAIL','passed':bool(passed),'evidence':C.file_record(path),'scope':scope}



def run_input_pipeline(root,artifact,req,image_size,deadline):
    P=C.pipeline(root);R=C.runner(root);folder=Path(artifact)/'phases/feasibility'/('resolution'+str(image_size));folder.mkdir(parents=True,exist_ok=True)
    model,config,source=P.learned_source(root,'vit',req['source_run']);model.cpu();del model
    config=copy.deepcopy(config);config.update(image_size=int(image_size),device='cpu',cpu_threads=2,dtype='float32');config['training'].update(microbatch=16,accumulation_steps=8)
    # CPU-only configuration keeps the parent free of CUDA context before the
    # independent operator case subprocesses check foreign compute activity.
    bundle=R.data_bundle(root,folder/'data',config,verify_images=True);path=folder/'pipeline_result.json'
    if path.exists():pipeline=C.read(path)
    else:
        status(artifact,'RUNNING','shared_train_only_input_pipeline_'+str(image_size));pipeline=P.benchmark_input_pipeline(bundle,config,deadline,workers=(0,4,8),repeats=3)
        pipeline.update(source_hashes=req['sources']);C.atomic_json(path,pipeline)
    if pipeline.get('source_hashes')!=req['sources']or pipeline.get('status')!='PASS':raise ValueError('Shared input pipeline receipt changed or failed')
    return bundle,config,pipeline,folder

def run_feasibility(root,artifact,req,clock):
    B,P,R,F=_modules(root);folder=Path(artifact)/'phases/feasibility';folder.mkdir(parents=True,exist_ok=True);deadline=clock['deadline_unix']
    summary_path=folder/'feasibility_result.json'
    if summary_path.exists():return C.read(summary_path)
    C.check_deadline(deadline,30)
    first_bundle,first_config,first_pipeline,first_folder=run_input_pipeline(root,artifact,req,256,deadline)
    pipeline_record=C.file_record(first_folder/'pipeline_result.json')
    status(artifact,'RUNNING','train_only_frozen_operator_cases')
    bench=B.compression_benchmark(root,folder/'operator',deadline=deadline,profile='full',handoff=req['reference'])
    operator=_receipt('operator',folder/'operator/compression_benchmark.json',bench.get('status')=='PASS'and bench.get('passed',False),
        'Complete trained-token finite-kernel correctness and all-case empirical softmax output/gradient fidelity; ranks32/64/128, N256/1024/4096')
    if not operator['passed']:
        result={'status':'NO_GO'if bench.get('status')=='NO_GO'else'UNKNOWN','reason':'Fixed kernel correctness/fidelity cases did not produce one complete eligible candidate',
            'gate_receipts':[operator],'official_test':'CLOSED','benchmark':C.file_record(folder/'operator/compression_benchmark.json'),'pipeline':pipeline_record}
        C.atomic_json(summary_path,result);return result
    selected=copy.deepcopy(bench['selected']);costs=[];pipeline_records=[]
    for image_size in(256,512):
        C.check_deadline(deadline,60);costfolder=folder/('resolution'+str(image_size));costfolder.mkdir(parents=True,exist_ok=True)
        if image_size==256:bundle,config,pipeline=first_bundle,first_config,first_pipeline
        else:bundle,config,pipeline,costfolder=run_input_pipeline(root,artifact,req,image_size,deadline)
        pipepath=costfolder/'pipeline_result.json'
        workers=int(pipeline['selected_workers']);P.install_input_pipeline(workers,root);pipeline_records.append(C.file_record(pipepath))
        proofpath=costfolder/'preflight_result.json'
        preflight_planpath=costfolder/'preflight_plan.json'
        if preflight_planpath.exists():preflight_plan=C.read(preflight_planpath)
        else:
            proposal={'seeds':[20261890+image_size],'schedule_updates':4000,'active_wall_seconds_per_arm':900.,'hard_limit_seconds':7200.,'image_size':image_size}
            preflight_plan=R.make_plan(root,req['run_id'],proposal,selected,devices=('cuda:0',),stage_id='feasibility-preflight'+str(image_size),input_workers=workers,deadline=deadline,gate_receipts=[operator])
            C.immutable(preflight_planpath,preflight_plan)
        status(artifact,'RUNNING','actual_full_state_preflight_'+str(image_size))
        proof=R.preflight(root,costfolder/'preflight',preflight_plan,bundle,deadline);C.atomic_json(proofpath,proof)
        if proof.get('status')!='PASS'or not all(v.get('status')=='PASS'and v.get('bitwise_equal')for v in proof.get('arms',{}).values()):raise ValueError('Actual registered micro16/accum8 exact checkpoint resume failed')
        status(artifact,'RUNNING','complete_jepa_cost_'+str(image_size))
        try:cost=B.full_jepa_speed_gate(root,costfolder,selected,deadline=deadline,image_size=image_size,microbatch=16,repetitions=5,preflight=proof,workers=workers,profile='full')
        except BaseException as exc:
            if isinstance(exc,TimeoutError):raise
            cost={'status':'UNKNOWN','passed':False,'reason':repr(exc),'image_size':image_size,'selected':selected,'source_hashes':req['sources']}
            C.atomic_json(costfolder/'full_jepa_speed_gate.json',cost)
        costs.append(C.file_record(costfolder/'full_jepa_speed_gate.json'))
        if cost.get('status')=='PASS'and cost.get('passed')and cost.get('memory_no_increase')and cost.get('measured_end_to_end_speed_over_sdpa',0)>=1.1:
            chosen={**selected,'image_size':image_size};cost_receipt=_receipt('full_jepa_cost',costfolder/'full_jepa_speed_gate.json',True,'Alternating full JEPA steps on a GPU observed without foreign compute; same real-data pipeline, FP32, effective128/physical16, exact state resume; >=1.1x speed and no CUDA-memory increase')
            result={'status':'PASS','selected':chosen,'input_workers':workers,'gate_receipts':[operator,cost_receipt],'benchmark':C.file_record(folder/'operator/compression_benchmark.json'),
                'cost':costs[-1],'pipeline':C.file_record(pipepath),'all_cost_receipts':costs,'all_pipeline_receipts':pipeline_records,'official_test':'CLOSED'}
            C.atomic_json(summary_path,result);return result
        if cost.get('status')=='UNKNOWN':
            # Configuration integrity/numerical failures are not circumvented by trying another resolution.
            result={'status':'UNKNOWN','reason':'Complete JEPA cost measurement did not complete validly','selected':selected,'gate_receipts':[operator],
                'benchmark':C.file_record(folder/'operator/compression_benchmark.json'),'all_cost_receipts':costs,'official_test':'CLOSED'};C.atomic_json(summary_path,result);return result
    result={'status':'NO_GO','reason':'No registered256/512px resolution gave>=10% complete JEPA throughput gain with no memory increase',
        'selected':selected,'gate_receipts':[operator],'benchmark':C.file_record(folder/'operator/compression_benchmark.json'),'all_cost_receipts':costs,'all_pipeline_receipts':pipeline_records,'official_test':'CLOSED'}
    C.atomic_json(summary_path,result);return result



def verify_feasibility(root,req,value):
    if value.get('status')!='PASS':raise ValueError('Later phases require actual completed feasibility PASS')
    for role in('benchmark','cost','pipeline'):
        record=value[role]
        if C.file_record(record['path'])!=record:raise ValueError('Frozen feasibility evidence changed: '+role)
    benchmark=C.read(value['benchmark']['path']);cost=C.read(value['cost']['path']);pipeline=C.read(value['pipeline']['path'])
    if any(v.get('source_hashes')!=req['sources']for v in(benchmark,cost,pipeline)):raise ValueError('Feasibility evidence source hashes differ')
    if benchmark.get('status')!='PASS'or not benchmark.get('passed')or cost.get('status')!='PASS'or not cost.get('passed'):raise ValueError('Actual operator or full JEPA gate did not pass')
    selected=value['selected'];options={k:selected[k]for k in('rank','feature_seed','local_radius','spatial_mode')}
    if options!=benchmark['selected']or options!={k:cost['selected'][k]for k in options}:raise ValueError('Selected candidate differs from actual audited candidate')
    if int(selected['image_size'])!=int(cost['image_size'])or value['input_workers']!=pipeline['selected_workers']:raise ValueError('Selected resolution or shared pipeline differs')
    if cost.get('measured_end_to_end_speed_over_sdpa',0)<1.1 or not cost.get('memory_no_increase'):raise ValueError('Complete JEPA speed or memory gate failed')
    verify_gate_receipts(root,value['gate_receipts']);return benchmark,cost,pipeline


def verify_gate_receipts(root,receipts):
    for gate in receipts:
        if gate.get('status')!='PASS'or not gate.get('passed')or gate.get('fixture_source'):raise ValueError('Required actual gate has no PASS')
        record=gate['evidence'];path=Path(record['path']).resolve()
        if not path.is_relative_to(Path(root).resolve()/'artifacts'/'notebook_17')or C.file_record(path)!=record:raise ValueError('Actual gate evidence checksum/path changed')
    return True


def partial_evidence(artifact):
    artifact=Path(artifact);operator=artifact/'phases/feasibility/operator';bench=None
    complete=operator/'compression_benchmark.json';progress=operator/'compression_progress.json'
    if complete.exists():bench=C.read(complete)
    elif progress.exists():
        current=C.read(progress);bench={'status':'UNKNOWN','passed':False,'identity':current.get('identity',{}),'rows':list(current.get('completed',{}).values()),'selected':None,'incomplete_audit':True}
    return bench

def _phase_worker_folder(artifact,phase,worker):return Path(artifact)/'phases'/phase/'workers'/worker



def phase_devices(root,folder):
    folder=Path(folder);path=folder/'device_selection.json'
    if path.exists():
        saved=C.read(path)
        if saved.get('source_hashes')!=C.source_hashes(root):raise ValueError('Frozen phase device observation source changed')
        return saved['devices']
    B=C.load(root,'experiments/17_fixed_kernel/benchmark.ipynb','fmm17_workflow_benchmark');available=[];observations={}
    indices=subprocess.check_output(['nvidia-smi','--query-gpu=index','--format=csv,noheader'],text=True).splitlines()
    for line in indices:
        device='cuda:'+str(int(line.strip()))
        try:observations[device]=B._exclusive_gpu(device);available.append(device)
        except RuntimeError as exc:observations[device]={'status':'UNAVAILABLE_OR_TELEMETRY_UNKNOWN','reason':repr(exc)}
    if not available:raise RuntimeError('No GPU observed free of foreign compute; existing project jobs are not stopped')
    saved={'status':'FROZEN_AVAILABLE_DEVICES','devices':available,'observations':observations,'source_hashes':C.source_hashes(root),
        'policy':'Only observed GPUs without foreign active compute enter this phase; same frozen device per seed pair; no other jobs terminated','observed_unix':time.time()}
    C.immutable(path,saved);return available


def worker_names(plan):
    devices=plan.get('devices')or list(dict.fromkeys(a['config']['device']for a in plan['arms']))
    return tuple('worker'+str(i)for i in range(len(devices)))

def run_worker(root,run_id,phase,worker):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact)
    if clock['phase']!=phase:raise ValueError('Worker phase clock differs from current activation')
    folder=_phase_worker_folder(artifact,phase,worker);R=C.runner(root);plan=C.read(folder/'plan.json')
    reserve=float(C.read(Path(artifact)/'phases'/phase/'worker_policy.json')['evaluation_reserve_seconds'])
    try:
        status(folder,'RUNNING','verify_data_and_resume');C.environment(plan['arms'][0]['config']);bundle=R.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=False)
        training=R.train_worker(root,folder,plan,bundle,clock['deadline_unix']-reserve)
        if training.get('status')!='COMPLETE':status(folder,'UNKNOWN','training_fairness');return training
        status(folder,'RUNNING','converged_readouts');evaluation=R.evaluate_worker(root,folder,plan,bundle,training,clock['deadline_unix'])
        state='COMPLETE'if evaluation.get('status')=='COMPLETE'else'UNKNOWN';status(folder,state,'worker_finished');return {'status':state,'training':str(folder/'training_result.json'),'evaluation':str(folder/'evaluation_result.json')}
    except BaseException as exc:
        status(folder,'UNKNOWN','worker_exception',reason=repr(exc),traceback=traceback.format_exc());raise


def launch_wait_workers(root,artifact,req,clock,plan,reserve):
    phase=clock['phase'];sessions=session_names(req['run_id']);R=C.runner(root);phasefolder=Path(artifact)/'phases'/phase;workers=worker_names(plan)
    C.immutable(phasefolder/'worker_policy.json',{'evaluation_reserve_seconds':float(reserve),'deadline_unix':clock['deadline_unix'],'phase':phase})
    for worker in workers:
        folder=_phase_worker_folder(artifact,phase,worker);folder.mkdir(parents=True,exist_ok=True)
        child=R.worker_plan(plan,worker);child['stage_id']=phase+'-'+worker;C.immutable(folder/'plan.json',child)
        existing=C.read(folder/'study_status.json')if(folder/'study_status.json').exists()else{}
        if existing.get('status')in('FAILED','UNKNOWN'):raise RuntimeError('Existing worker UNKNOWN; preserve timed fairness evidence, never reset')
        if existing.get('status')=='COMPLETE':continue
        startpath=folder/'launch_time.json';C.immutable(startpath,{'started_unix':C.read(startpath)['started_unix']if startpath.exists()else time.time()})
        notebook=write_operator_notebook(root,folder/'worker.ipynb','run_worker',{'root':str(root),'run_id':req['run_id'],'phase':phase,'worker':worker})
        spawn(root,notebook,sessions[worker],folder/'worker.log')
    while True:
        C.check_deadline(clock['deadline_unix']);states={w:C.read(_phase_worker_folder(artifact,phase,w)/'study_status.json')if(_phase_worker_folder(artifact,phase,w)/'study_status.json').exists()else{'status':'STARTING'}for w in workers}
        if any(v['status']in('FAILED','UNKNOWN')for v in states.values()):raise RuntimeError('Worker failed or cutoff: '+str({w:s['status']for w,s in states.items()}))
        if all(v['status']=='COMPLETE'for v in states.values()):break
        for worker,value in states.items():
            if value['status']in('STARTING','RUNNING')and time.time()-C.read(_phase_worker_folder(artifact,phase,worker)/'launch_time.json')['started_unix']>90 and not session_exists(sessions[worker]):raise RuntimeError('Worker tmux session disappeared; no unmeasured timing reset')
        C.atomic_json(Path(artifact)/'coordinator_heartbeat.json',{'updated_unix':time.time(),'phase':phase,'worker_states':states,'deadline_unix':clock['deadline_unix']});time.sleep(10)
    # All model/evaluation receipts are committed before worker completion. Free exact
    # session names now so the following phase cannot inherit a departing old worker.
    for worker in workers:terminate_session_tree(sessions[worker])
    training={'status':'COMPLETE','fair_wall_budget_verified':True,'arms':{},'gate_receipts':plan['gate_receipts']};rows=[];initial_rows=[]
    for worker in workers:
        folder=_phase_worker_folder(artifact,phase,worker);child=C.read(folder/'training_result.json');evaluation=C.read(folder/'evaluation_result.json')
        if child.get('status')!='COMPLETE'or not child.get('fair_wall_budget_verified')or evaluation.get('status')!='COMPLETE':raise ValueError('Completed worker receipt lacks fairness or convergence')
        training['arms'].update(child['arms']);rows.extend(evaluation['rows']);initial_rows.extend(evaluation.get('initial_rows',[]))
    values=list(training['arms'].values());tolerance=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
    if len(values)!=len(plan['arms'])or any(v.get('status')!='COMPLETE'or not v.get('fair_wall_budget_verified')for v in values)or max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)>tolerance:raise RuntimeError('Global registered active-wall fairness failed; official test remains closed')
    C.atomic_json(phasefolder/'training_result.json',training);evaluation={'status':'COMPLETE','rows':rows,'initial_rows':initial_rows,'official_test':'CLOSED'};C.atomic_json(phasefolder/'evaluation_result.json',evaluation)
    return training,evaluation


def analyze_screen(plan,evaluation):
    rows=evaluation.get('rows',[]);initial=evaluation.get('initial_rows',[]);required={(int(s),v)for s in plan['seeds']for v in plan['variants']}
    for group in(rows,initial):
        if len(group)!=len(required)or {(int(r['seed']),r['variant'])for r in group}!=required:return {'status':'UNKNOWN','quality_pass':False,'reason':'Missing or duplicate paired initial/final endpoints'}
        if any(r.get('status')!='COMPLETE'or not r.get('converged')or not r.get('numerical_valid',False)or r.get('fixture_source')for r in group):return {'status':'UNKNOWN','quality_pass':False,'reason':'Endpoint convergence/numerical validity not certified'}
        if any(not math.isfinite(float(r.get('accuracy_percent',float('nan'))))or not math.isfinite(float(r.get('knn_accuracy_percent',float('nan'))))for r in group):return {'status':'UNKNOWN','quality_pass':False,'reason':'Nonfinite accuracy or kNN endpoint'}
    final={(int(r['seed']),r['variant']):r for r in rows};start={(int(r['seed']),r['variant']):r for r in initial};pairs=[]
    for seed in plan['seeds']:
        baseline=final[(int(seed),'vit')];candidate=final[(int(seed),'optional_geometry')]
        pairs.append({'seed':int(seed),'vit_percent':baseline['accuracy_percent'],'candidate_percent':candidate['accuracy_percent'],
            'delta_pp':candidate['accuracy_percent']-baseline['accuracy_percent'],
            'vit_gain_from_initial_pp':baseline['accuracy_percent']-start[(int(seed),'vit')]['accuracy_percent'],
            'candidate_gain_from_initial_pp':candidate['accuracy_percent']-start[(int(seed),'optional_geometry')]['accuracy_percent'],
            'knn_delta_pp':candidate['knn_accuracy_percent']-baseline['knn_accuracy_percent']})
    effects=np.asarray([p['delta_pp']for p in pairs],dtype=float);mean=float(effects.mean());positive=int((effects>0).sum());sd=float(effects.std(ddof=1));ci=None if sd==0 else list(map(float,stats.t.interval(.95,len(effects)-1,loc=mean,scale=sd/math.sqrt(len(effects)))))
    vg=float(np.mean([p['vit_gain_from_initial_pp']for p in pairs]));cg=float(np.mean([p['candidate_gain_from_initial_pp']for p in pairs]));knn=float(np.mean([p['knn_delta_pp']for p in pairs]))
    passed=mean>=.5 and positive>=3 and vg>=.25 and cg>=.25 and knn>=-.5
    return {'status':'COMPLETE','quality_pass':passed,'decision':'PROMISING_SCREEN'if passed else'NO_GO','paired_results':pairs,'paired_seed_deltas_pp':effects.tolist(),
        'mean_delta_pp':mean,'raw_ci95_pp':ci,'positive_pairs':positive,'vit_gain_from_initial_pp':vg,'candidate_gain_from_initial_pp':cg,'knn_mean_delta_pp':knn,
        'gate_thresholds':{'mean_delta_pp':.5,'positive_pairs':3,'each_architecture_mean_gain_from_initial_pp':.25,'knn_mean_delta_pp':-.5},
        'inferential_unit':'Paired training seed; four-pair screening is not final statistical confirmation','official_test':'CLOSED'}


def run_screen(root,artifact,req,clock,feasibility):
    phase=clock['phase'];folder=Path(artifact)/'phases'/phase;folder.mkdir(parents=True,exist_ok=True);R=C.runner(root);planpath=folder/'plan.json'
    devices=phase_devices(root,folder)
    plan=C.read(planpath)if planpath.exists()else R.make_screen_plan(root,req['run_id'],feasibility['selected'],devices=devices,deadline=clock['deadline_unix'],input_workers=feasibility['input_workers'],gate_receipts=feasibility['gate_receipts'])
    C.immutable(planpath,plan);status(artifact,'RUNNING','four_fresh_pairs_equal_wall_learning',image_size=feasibility['selected']['image_size'],active_wall_seconds_per_arm=900)
    summary_path=folder/'screen_result.json'
    if summary_path.exists():screen=C.read(summary_path)
    else:
        training,evaluation=launch_wait_workers(root,artifact,req,clock,plan,reserve=2400. if phase=='screen512'else 1800.)
        screen=analyze_screen(plan,evaluation);C.atomic_json(summary_path,screen)
    handoff={'status':screen['status'],'quality_pass':bool(screen.get('quality_pass')),'candidate_variant':'optional_geometry','candidate_display':'Fixed positive content features plus exact3x3 local correction',
        'selected':feasibility['selected'],'candidate_plan':str(planpath),'plan_sha256':C.file_record(planpath)['sha256'],'training_result':str(folder/'training_result.json'),
        'training_sha256':C.file_record(folder/'training_result.json')['sha256'],'evaluation_result':str(folder/'evaluation_result.json'),
        'evaluation_sha256':C.file_record(folder/'evaluation_result.json')['sha256'],'screening':screen,'source_hashes':req['sources'],'fixture_source':None,'official_test':'CLOSED'}
    C.immutable(folder/'screen_handoff.json',handoff);quality=_receipt('quality',folder/'screen_result.json',handoff['quality_pass'],'Four predeclared fresh pairs; +0.5pp mean, >=3positive pairs, both own-initial gains, stable train-only fitted probes, kNN guard')
    return plan,screen,handoff,[*feasibility['gate_receipts'],quality]


def run_confirmation(root,artifact,req,clock,feasibility,screen_handoff,receipts):
    B,P,R,F=_modules(root);folder=Path(artifact)/'phases/confirmation';folder.mkdir(parents=True,exist_ok=True);deadline=clock['deadline_unix']
    devices=phase_devices(root,folder)
    bench=C.read(feasibility['benchmark']['path']);proposal=F.prepare_registration(root,folder,screen_handoff,bench,deadline-time.time(),configuration={'gpu_workers':len(devices)})
    if proposal.get('status')!='READY_FOR_FROZEN_CONFIRMATION':return {'status':'UNKNOWN','reason':'Fresh paired power/budget requirement is infeasible inside original confirmation phase','proposal':proposal,'gate_receipts':receipts,'official_test':'CLOSED'}
    planpath=folder/'plan.json'
    if planpath.exists():plan=C.read(planpath)
    else:
        proposal={**proposal,'image_size':feasibility['selected']['image_size'],'hard_limit_seconds':43200.}
        plan=R.make_plan(root,req['run_id'],proposal,feasibility['selected'],devices=devices,stage_id='confirmation',input_workers=feasibility['input_workers'],deadline=deadline,gate_receipts=receipts)
        C.immutable(planpath,plan)
    config=plan['arms'][0]['config'];bundle=R.data_bundle(root,folder/'data',config,verify_images=True)
    F.register_confirmation(root,folder,plan,screen_handoff['screening']['paired_seed_deltas_pp'],bundle)
    status(artifact,'RUNNING','fresh_power_derived_confirmation_pairs',pairs=len(plan['seeds']),active_wall_seconds_per_arm=plan['active_wall_seconds_per_arm'])
    training,evaluation=launch_wait_workers(root,artifact,req,clock,plan,float(proposal['evaluation_reserve_seconds']))
    status(artifact,'RUNNING','sealed_official_readout')
    result=F.seal_and_score(root,folder,plan,bundle,training,{'model_builder':R.builder(root,feasibility['selected']),'rows':evaluation['rows']},deadline)
    result.update(gate_receipts=receipts,source_hashes=req['sources'],selected=feasibility['selected'],screen_handoff=C.file_record(Path(artifact)/'phases'/('screen512'if feasibility['selected']['image_size']==512 else'screen256')/'screen_handoff.json'))
    return result


def run_coordinator(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);bench=None;cost=None;pipeline=None;screen=None
    from fmm_jepa.food101_training import run_writer
    try:
        with run_writer(artifact/'coordinator_writer'):
            phase=clock['phase'];feasibility_path=artifact/'phases/feasibility/feasibility_result.json'
            if phase=='feasibility':
                feasibility=run_feasibility(root,artifact,req,clock)
                if feasibility.get('status')!='PASS':
                    report(root,artifact,feasibility,bench=C.read(feasibility['benchmark']['path'])if feasibility.get('benchmark')else None,pipeline=C.read(feasibility['pipeline']['path'])if feasibility.get('pipeline')else None);status(artifact,feasibility['status'],'feasibility_finished');return feasibility
                clock=activate_phase(root,artifact,'screen512'if feasibility['selected']['image_size']==512 else'screen256')
            else:feasibility=C.read(feasibility_path)
            if feasibility.get('status')!='PASS':raise ValueError('Later phase lacks PASS feasibility')
            bench,cost,pipeline=verify_feasibility(root,req,feasibility)
            if clock['phase'].startswith('screen'):
                plan,screen,handoff,receipts=run_screen(root,artifact,req,clock,feasibility)
                report(root,artifact,{'status':'SCREEN_COMPLETE','screen':screen,'gate_receipts':receipts,'official_test':'CLOSED'},bench,cost,pipeline,screen)
                if screen.get('status')!='COMPLETE'or not screen.get('quality_pass'):
                    result={'status':'NO_GO'if screen.get('status')=='COMPLETE'else'UNKNOWN','reason':'Four-pair screen did not meet predeclared promising-learning gate','screen':screen,'gate_receipts':receipts,'official_test':'CLOSED'}
                    report(root,artifact,result,bench,cost,pipeline,screen);status(artifact,result['status'],'learning_screen_finished');return result
                clock=activate_phase(root,artifact,'confirmation')
            else:
                screen_phase='screen512'if feasibility['selected']['image_size']==512 else'screen256';screen=C.read(artifact/'phases'/screen_phase/'screen_result.json');handoff=C.read(artifact/'phases'/screen_phase/'screen_handoff.json')
                quality=_receipt('quality',artifact/'phases'/screen_phase/'screen_result.json',screen.get('quality_pass',False),'Frozen actual17 four-pair promising-learning screen');receipts=[*feasibility['gate_receipts'],quality]
            verify_gate_receipts(root,receipts)
            result=run_confirmation(root,artifact,req,clock,feasibility,handoff,receipts);report(root,artifact,result,bench,cost,pipeline,screen);status(artifact,result['status'],'finished',report=str(artifact/'report.html'));return result
    except BaseException as exc:
        for worker in('worker0','worker1'):terminate_session_tree(session_names(run_id)[worker])
        result={'status':'UNKNOWN','reason':repr(exc),'traceback':traceback.format_exc(),'official_test':'CLOSED_UNLESS_VALID_EXISTING_SEAL','source_hashes':req['sources']}
        report(root,artifact,result,bench or partial_evidence(artifact),cost,pipeline,screen);status(artifact,'UNKNOWN','cutoff_or_integrity_failure',reason=repr(exc));return result


def run_guard(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);sessions=session_names(run_id);last_review=0.;missing_since=None
    try:
        while True:
            req,clock=check_registration(root,artifact);existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if existing.get('status')in TERMINAL:return {'status':'GUARD_COMPLETE','study_status':existing['status']}
            now=time.time()
            if not session_exists(sessions['coordinator']):
                if missing_since is None:missing_since=now
                elif now-missing_since>90:raise RuntimeError('Coordinator absent beyond startup grace; stop all descendants and preserve UNKNOWN')
            else:missing_since=None
            if now>=clock['deadline_unix']:raise TimeoutError('Original '+clock['phase']+' phase clock exhausted')
            C.atomic_json(artifact/'guard_heartbeat.json',{'status':'HEALTHY','updated_unix':now,'phase':clock['phase'],'deadline_unix':clock['deadline_unix'],'sessions':{k:session_exists(v)for k,v in sessions.items()}})
            if now-last_review>=3600:
                runner,_,_=C.modules(root);audits=[]
                for manifest in(root/'checkpoints'/run_id).glob('stages/*/arms/*/manifest.json'):
                    current=validate_manifest_rotation(runner,manifest.parent);audits.append({'directory':str(manifest.parent),'successful_updates':current['successful_updates'],'latest_valid':True})
                C.atomic_json(artifact/'hourly_review.json',{'status':'PASS','updated_unix':now,'phase':clock['phase'],'audits':audits,'source_and_original_clock_valid':True,'official_test':'No predictive data reader in guard; final seal controls opening'});last_review=now
            time.sleep(10)
    except BaseException as exc:
        stopped={role:terminate_session_tree(sessions[role])for role in('coordinator','worker0','worker1')}
        result={'status':'UNKNOWN','reason':repr(exc),'stopped_project_processes':stopped,'official_test':'CLOSED_UNLESS_VALID_EXISTING_SEAL','active_phase_clock':locals().get('clock',{'status':'UNREADABLE_ORIGINAL_CLOCK'})}
        report(root,artifact,result,bench=partial_evidence(artifact));status(artifact,'UNKNOWN','guard_deadline_or_integrity_failure',reason=repr(exc));return result